# EDA: Proyección de Gastos a Fin de Mes
Estima cuánto gastará el usuario al final del mes basándose en su tasa de gasto de los últimos 15 días, y calcula el balance restante.

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta
import calendar
from pathlib import Path

BASE_TXN = Path("../../Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})
df_productos = pd.read_csv(BASE_TXN / "hey_productos.csv", dtype={"producto_id": str, "user_id": str})
df_transacc = pd.read_csv(BASE_TXN / "hey_transacciones.csv", dtype={"transaccion_id": str, "user_id": str, "producto_id": str}, parse_dates=["fecha_hora"])


## 1. Carga Fija y Variables de Tiempo

In [2]:
mensualidades = (df_productos[df_productos["monto_mensualidad"].notna()]
                 .groupby("user_id")["monto_mensualidad"].sum()
                 .rename("total_mensualidades").to_frame())
recurrentes = (df_transacc[df_transacc["tipo_operacion"] == "cargo_recurrente"]
               .groupby("user_id")["monto"].sum()
               .rename("total_recurrentes").to_frame())
carga = mensualidades.join(recurrentes, how="outer").fillna(0)
carga["carga_fija_total"] = carga["total_mensualidades"] + carga["total_recurrentes"]

hoy = df_transacc["fecha_hora"].max()
hace_15 = hoy - timedelta(days=15)
inicio_mes_actual = hoy.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
dias_en_mes = calendar.monthrange(hoy.year, hoy.month)[1]
dias_restantes_mes = dias_en_mes - hoy.day

print(f"Fecha máxima en dataset: {hoy}")
print(f"Días restantes del mes ({hoy.month}/{hoy.year}): {dias_restantes_mes}")


Fecha máxima en dataset: 2025-12-15 13:59:36
Días restantes del mes (12/2025): 16


## 2. Gasto Reciente y Acumulado

In [3]:
df_compras = df_transacc[(df_transacc["tipo_operacion"]=="compra") & (df_transacc["estatus"]=="completada")]

gasto_reciente = (df_compras[df_compras["fecha_hora"] >= hace_15]
                  .groupby("user_id")["monto"].sum()
                  .rename("gasto_ultimos_15_dias").to_frame())

gasto_mes_actual = (df_compras[df_compras["fecha_hora"] >= inicio_mes_actual]
                    .groupby("user_id")["monto"].sum()
                    .rename("gasto_acumulado_mes").to_frame())

inicio_mes_anterior = (inicio_mes_actual - timedelta(days=1)).replace(day=1)
gasto_mes_anterior = (df_compras[(df_compras["fecha_hora"] >= inicio_mes_anterior) & (df_compras["fecha_hora"] < inicio_mes_actual)]
                      .groupby("user_id")["monto"].sum()
                      .rename("gasto_real_mes_anterior").to_frame())

historial = df_transacc.groupby("user_id")["fecha_hora"].min().rename("primera_txn").to_frame()
historial["historial_mayor_15d"] = historial["primera_txn"] <= hace_15


## 3. Proyección y Déficit Estimado

In [4]:
df_proy = df_clientes[["user_id", "ingreso_mensual_mxn"]].set_index("user_id")
df_proy = df_proy.join(carga[["carga_fija_total"]], how="left").fillna(0)
df_proy = df_proy.join(gasto_reciente, how="left").fillna({"gasto_ultimos_15_dias": 0})
df_proy = df_proy.join(gasto_mes_actual, how="left").fillna({"gasto_acumulado_mes": 0})
df_proy = df_proy.join(gasto_mes_anterior, how="left").fillna({"gasto_real_mes_anterior": 0})
df_proy = df_proy.join(historial[["historial_mayor_15d"]], how="left").fillna({"historial_mayor_15d": False})

df_proy = df_proy[df_proy["historial_mayor_15d"]].copy()

df_proy["tasa_gasto_diario"] = df_proy["gasto_ultimos_15_dias"] / 15
df_proy["gasto_estimado_fin_mes"] = df_proy["gasto_acumulado_mes"] + (df_proy["tasa_gasto_diario"] * dias_restantes_mes)
df_proy["ingreso_restante_estimado"] = df_proy["ingreso_mensual_mxn"] - df_proy["gasto_estimado_fin_mes"] - df_proy["carga_fija_total"]

df_proy["balance_actual"] = df_proy["ingreso_mensual_mxn"] - df_proy["carga_fija_total"] - df_proy["gasto_acumulado_mes"]
df_proy["dias_hasta_deficit"] = np.where(
    df_proy["balance_actual"] <= 0, 0,
    np.where(df_proy["tasa_gasto_diario"] > 0, df_proy["balance_actual"] / df_proy["tasa_gasto_diario"], np.inf)
)

df_proy.head()


,ingreso_mensual_mxn,carga_fija_total,gasto_ultimos_15_dias,gasto_acumulado_mes,gasto_real_mes_anterior,historial_mayor_15d,tasa_gasto_diario,gasto_estimado_fin_mes,ingreso_restante_estimado,balance_actual,dias_hasta_deficit
user_id,,,,,,,,,,,
USR-00001,24500,3891.0,0.0,0.0,1154.08,True,0.0,0.0,20609.0,20609.0,inf
USR-00002,19000,4688.0,0.0,0.0,1396.74,True,0.0,0.0,14312.0,14312.0,inf
USR-00003,14000,3542.0,0.0,0.0,823.49,True,0.0,0.0,10458.0,10458.0,inf
USR-00004,61000,2195.0,0.0,0.0,2338.12,True,0.0,0.0,58805.0,58805.0,inf
USR-00005,27000,5583.0,0.0,0.0,1444.74,True,0.0,0.0,21417.0,21417.0,inf


In [5]:
print(f"Usuarios procesados (con historial >15d): {len(df_proy)}")
usuarios_deficit = (df_proy["ingreso_restante_estimado"] < 0).sum()
print(f"Usuarios con deficit proyectado a fin de mes: {usuarios_deficit} ({(usuarios_deficit/len(df_proy))*100:.2f}%)")

mae = (df_proy["gasto_estimado_fin_mes"] - df_proy["gasto_real_mes_anterior"]).abs().mean()
print(f"\nValidación Cruzada | MAE (Proyección actual vs Gasto Real Mes Anterior): {mae:.2f}")


Usuarios procesados (con historial >15d): 15025
Usuarios con deficit proyectado a fin de mes: 40 (0.27%)

Validación Cruzada | MAE (Proyección actual vs Gasto Real Mes Anterior): 2780.69


## 4. Guardar Resultados en CSV

In [6]:
output_dir = Path("../../outputs")
output_dir.mkdir(exist_ok=True)
out_path = output_dir / "proyeccion_gastos_fin_mes.csv"
df_proy.to_csv(out_path)
print("Archivo CSV generado exitosamente en:", out_path.resolve())


Archivo CSV generado exitosamente en: D:\Datamoles\outputs\proyeccion_gastos_fin_mes.csv
